# Register AzureML Model & Serve as Endpoint

This notebook runs **inside Microsoft Fabric**. It loads model artifacts that were uploaded
to a Lakehouse (by the AzureML deploy notebook), registers the model with MLflow
(including a proper signature and input example), and activates a real-time endpoint.

### Before you start
1. **Attach a Lakehouse** to this notebook (left panel → Lakehouses → Add)
2. Verify model artifacts exist at `Files/azureml_models/<model_name>/` in your Lakehouse
3. Update the `MODEL_NAME` and `MODEL_PATH` in the config cell below

### What this notebook does
1. Copies model artifacts from OneLake to a local temp directory
2. Loads the model and **infers the input/output signature** (required for endpoints)
3. Registers the model in Fabric's MLflow registry with the signature + input example
4. Verifies the model has proper schemas
5. Activates the model endpoint for real-time scoring
6. Tests the endpoint with sample data

## Configuration

Update these values to match your model and Lakehouse.

In [ ]:
# ---- Update these values ----
MODEL_NAME = "credit_defaults_model"

# Path to model artifacts in your Lakehouse.
# Format: abfss://<workspace_id>@onelake.dfs.fabric.microsoft.com/<lakehouse_id>/Files/azureml_models/<model_name>
# Tip: you can find this by right-clicking the folder in the Lakehouse Files browser → "Copy ABFS path"
MODEL_PATH = "abfss://<workspace_id>@onelake.dfs.fabric.microsoft.com/<lakehouse_id>/Files/azureml_models/credit_defaults_model"

## Step 1: Copy Artifacts & Check Dependencies

The model was trained on AzureML with a specific scikit-learn version. Fabric's runtime may
have a different version. **Step 2 includes an automatic patch** for the most common issue
(GradientBoosting `_loss` attribute renamed between sklearn 1.0.x and 1.2.x). This cell copies
the artifacts and reports any version mismatch.

In [ ]:
import os
import tempfile
import yaml
import json
import re
from notebookutils import mssparkutils

# ---- Copy model artifacts from OneLake to local temp dir ----
local_dir = tempfile.mkdtemp()

def copy_recursive(src_path, dst_dir):
    entries = mssparkutils.fs.ls(src_path)
    for f in entries:
        if f.isDir:
            sub = os.path.join(dst_dir, f.name)
            os.makedirs(sub, exist_ok=True)
            copy_recursive(f.path, sub)
        else:
            mssparkutils.fs.cp(f.path, f"file://{dst_dir}/{f.name}")
            print(f"  copied {f.name}")

copy_recursive(MODEL_PATH, local_dir)
print(f"[OK] Model artifacts copied to {local_dir}")
print(f"Files: {os.listdir(local_dir)}")

## Step 2: Load Model & Infer Signature

In [ ]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import yaml
import json
import os
import shutil
from mlflow.models.signature import ModelSignature
from mlflow.models import infer_signature

def _patch_gradient_boosting_loss(model):
    """Fix sklearn 1.0.x -> 1.2.x pickle compatibility for GradientBoosting models.

    sklearn 1.2 renamed the internal loss_ attribute to _loss. Models pickled
    with sklearn 1.0.x have loss_ but lack _loss, causing AttributeError on
    predict(). This patches the attribute once during registration; the re-logged
    model will have a clean pickle created with the current sklearn version.
    """
    if 'GradientBoosting' not in type(model).__name__:
        return
    if hasattr(model, '_loss'):
        return

    # Approach 1: alias from the old attribute name (loss_ -> _loss)
    if hasattr(model, 'loss_') and model.loss_ is not None:
        model._loss = model.loss_
        print(f"  [FIX] Patched {type(model).__name__}._loss from loss_ (sklearn version migration)")
        return

    # Approach 2: reconstruct from _gb_losses (defensive fallback)
    from sklearn.ensemble._gb_losses import (
        BinomialDeviance, MultinomialDeviance, ExponentialLoss,
        LeastSquaresError, LeastAbsoluteError, HuberLossFunction, QuantileLossFunction,
    )
    is_clf = hasattr(model, 'classes_')
    loss_name = getattr(model, 'loss', 'deviance')
    if is_clf:
        n_classes = getattr(model, 'n_classes_', 2)
        if loss_name in ('deviance', 'log_loss'):
            model._loss = MultinomialDeviance(n_classes) if n_classes > 2 else BinomialDeviance(n_classes)
        elif loss_name == 'exponential':
            model._loss = ExponentialLoss(n_classes)
    else:
        if loss_name in ('huber',):
            model._loss = HuberLossFunction(getattr(model, 'alpha', 0.9))
        elif loss_name in ('quantile',):
            model._loss = QuantileLossFunction(getattr(model, 'alpha', 0.5))
        elif loss_name in ('absolute_error', 'lad'):
            model._loss = LeastAbsoluteError()
        else:
            model._loss = LeastSquaresError()
    print(f"  [FIX] Reconstructed {type(model).__name__}._loss (sklearn version migration)")

# ---- Parse MLmodel metadata ----
with open(os.path.join(local_dir, "MLmodel")) as fh:
    mlmodel = yaml.safe_load(fh)

flavors = mlmodel.get("flavors", {})
print(f"Model flavors: {list(flavors.keys())}")

# ---- Load the model ----
if "sklearn" in flavors:
    model_obj = mlflow.sklearn.load_model(local_dir)
    _patch_gradient_boosting_loss(model_obj)
elif "lightgbm" in flavors:
    import mlflow.lightgbm
    model_obj = mlflow.lightgbm.load_model(local_dir)
elif "xgboost" in flavors:
    import mlflow.xgboost
    model_obj = mlflow.xgboost.load_model(local_dir)
else:
    model_obj = mlflow.pyfunc.load_model(local_dir)

print(f"[OK] Model loaded: {type(model_obj).__name__}")

# ---- Build signature for Fabric endpoint ----
# Fabric requires ColSpec (column-based) signatures with 'double' types.
# We build the signature explicitly to ensure maximum compatibility.
from mlflow.types import Schema, ColSpec

n_features = getattr(model_obj, "n_features_in_", None)
feature_names = getattr(model_obj, "feature_names_in_", None)

# Determine column names: use original names, MLmodel signature, or generate
if feature_names is not None:
    col_names = [str(c) for c in feature_names]
else:
    # Try parsing column names from the original MLmodel signature
    sig_data = mlmodel.get("signature", {})
    inputs_str = sig_data.get("inputs", "[]")
    try:
        inputs_schema = json.loads(inputs_str) if inputs_str else []
    except (json.JSONDecodeError, TypeError):
        inputs_schema = []
    if inputs_schema and isinstance(inputs_schema[0], dict) and "name" in inputs_schema[0]:
        col_names = [col["name"] for col in inputs_schema]
    elif n_features is not None:
        col_names = [f"feature_{i}" for i in range(n_features)]
    else:
        col_names = [f"feature_{i}" for i in range(max(len(inputs_schema), 1))]

sample_input = pd.DataFrame(np.zeros((5, len(col_names))), columns=col_names)

# Verify model works after monkey-patch
predictions = model_obj.predict(sample_input)
print(f"[OK] predict() returned {len(predictions)} predictions")

# Build explicit ColSpec signature — use 'double' for output to avoid
# Fabric UI issues with integer ('long') schemas
input_schema = Schema([ColSpec("double", name) for name in col_names])
output_schema = Schema([ColSpec("double", "prediction")])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

print(f"[OK] Built ColSpec signature:")
print(f"  Inputs:  {len(col_names)} columns (double)")
print(f"  Outputs: 1 column 'prediction' (double)")

## Step 3: Register Model in MLflow

In [ ]:
mlflow.set_experiment("azureml-model-import")

with mlflow.start_run(run_name="azureml-import") as run:
    # Log basic metadata
    mlflow.log_param("source", "azureml")
    mlflow.log_param("model_type", type(model_obj).__name__)
    mlflow.log_param("n_features", sample_input.shape[1])

    # Log model with signature AND input_example (both are critical for endpoints)
    if "sklearn" in flavors:
        mlflow.sklearn.log_model(
            model_obj, "model",
            signature=signature,
            input_example=sample_input.iloc[:2]
        )
    elif "lightgbm" in flavors:
        mlflow.lightgbm.log_model(
            model_obj, "model",
            signature=signature,
            input_example=sample_input.iloc[:2]
        )
    elif "xgboost" in flavors:
        mlflow.xgboost.log_model(
            model_obj, "model",
            signature=signature,
            input_example=sample_input.iloc[:2]
        )
    else:
        raise RuntimeError(
            f"Unsupported flavor for endpoints: {list(flavors.keys())}. "
            "Fabric endpoints support: sklearn, lightgbm, xgboost, keras."
        )

    run_id = run.info.run_id
    print(f"[OK] Model logged to run: {run_id}")

# Register the model
mv = mlflow.register_model(f"runs:/{run_id}/model", MODEL_NAME)
print(f"[OK] Registered: {MODEL_NAME} v{mv.version}")

# Cleanup temp files
shutil.rmtree(local_dir)
print(f"\n>>> Model version {mv.version} is ready. Check the Fabric UI to verify schemas.")

## Step 4: Verify Schemas

Open the model in the Fabric UI and verify:
- **Input schema** count is > 0
- **Output schema** count is > 0

If both show 0, the model won't support endpoints. Re-check the signature inference above.

The cell below prints schema info from the MLflow registry.

In [ ]:
from mlflow import MlflowClient

client = MlflowClient()
model_version = client.get_model_version(MODEL_NAME, mv.version)

# Load the run to check logged model signature
run_data = client.get_run(run_id)
artifacts = client.list_artifacts(run_id, "model")
print(f"Model: {MODEL_NAME} v{mv.version}")
print(f"Run ID: {run_id}")
print(f"Status: {model_version.status}")
print(f"Artifacts: {[a.path for a in artifacts]}")
print(f"\nSignature (from inference):")
print(f"  Inputs:  {signature.inputs}")
print(f"  Outputs: {signature.outputs}")
print(f"\n✅ If you see inputs/outputs above, the model should support endpoints.")
print(f"Go to the Fabric UI → Models → {MODEL_NAME} → version {mv.version} to verify.")

## Step 5: Activate Endpoint

You can activate the endpoint in **two ways**:

**Option A (UI):** In the Fabric model page, click **"Activate version endpoint"** in the ribbon.

**Option B (Code):** Run the cell below to activate via the Fabric REST API.

In [ ]:
import time
import requests

# Get workspace context from the notebook environment
FABRIC_API_BASE = "https://api.fabric.microsoft.com/v1"

# Get token from the Fabric notebook environment
token = mssparkutils.credentials.getToken("pbi")
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Find the workspace ID from the MODEL_PATH
# MODEL_PATH format: abfss://<workspace_id>@onelake...
WORKSPACE_ID = MODEL_PATH.split("//")[1].split("@")[0]

# Find the model item ID
resp = requests.get(
    f"{FABRIC_API_BASE}/workspaces/{WORKSPACE_ID}/items",
    params={"type": "MLModel"},
    headers=headers,
)
resp.raise_for_status()
model_match = [m for m in resp.json().get("value", []) if m["displayName"] == MODEL_NAME]
if not model_match:
    raise ValueError(f"Model '{MODEL_NAME}' not found in Fabric.")

FABRIC_MODEL_ID = model_match[0]["id"]
MODEL_VERSION = str(mv.version)
print(f"[OK] Found model: {MODEL_NAME} ({FABRIC_MODEL_ID}), activating v{MODEL_VERSION}...")

# Activate endpoint
activate_url = (
    f"{FABRIC_API_BASE}/workspaces/{WORKSPACE_ID}"
    f"/mlmodels/{FABRIC_MODEL_ID}/endpoint/versions/{MODEL_VERSION}/activate"
)
resp = requests.post(activate_url, headers=headers)

if resp.status_code == 200:
    print("[OK] Endpoint activated!")
elif resp.status_code == 202:
    location = resp.headers.get("Location")
    if not location:
        raise RuntimeError("Got 202 but no Location header.")
    print("Activating (this takes a few minutes)...")
    for attempt in range(60):
        time.sleep(30)
        poll_resp = requests.get(location, headers={"Authorization": f"Bearer {mssparkutils.credentials.getToken('pbi')}"})
        poll_resp.raise_for_status()
        data = poll_resp.json()
        status = data.get("status", "Unknown")
        print(f"   [{attempt+1}] {status}")
        if status in ("Succeeded", "Completed"):
            print("[OK] Endpoint is now ACTIVE!")
            break
        if status == "Failed":
            raise RuntimeError(f"Activation failed: {json.dumps(data, indent=2)}")
else:
    resp.raise_for_status()

## Step 6: Test Scoring

In [ ]:
# Refresh token
headers = {"Authorization": f"Bearer {mssparkutils.credentials.getToken('pbi')}", "Content-Type": "application/json"}

score_url = (
    f"{FABRIC_API_BASE}/workspaces/{WORKSPACE_ID}"
    f"/mlmodels/{FABRIC_MODEL_ID}/endpoint/versions/{MODEL_VERSION}/score"
)

# Use the same sample data we used for signature inference
test_data = sample_input.iloc[:3].values.tolist()
payload = {
    "formatType": "dataframe",
    "orientation": "values",
    "inputs": test_data,
}

print(f"Scoring with {len(test_data)} rows...")
resp = requests.post(score_url, headers=headers, json=payload)

if resp.status_code == 200:
    print("[OK] Predictions:")
    print(json.dumps(resp.json(), indent=2))
elif resp.status_code == 202:
    print("Endpoint waking up (auto-sleep), waiting...")
    location = resp.headers.get("Location")
    if location:
        for attempt in range(30):
            time.sleep(30)
            poll = requests.get(location, headers={"Authorization": f"Bearer {mssparkutils.credentials.getToken('pbi')}"})
            if poll.json().get("status") in ("Succeeded", "Completed"):
                break
    # Retry scoring
    headers = {"Authorization": f"Bearer {mssparkutils.credentials.getToken('pbi')}", "Content-Type": "application/json"}
    resp2 = requests.post(score_url, headers=headers, json=payload)
    resp2.raise_for_status()
    print("[OK] Predictions:")
    print(json.dumps(resp2.json(), indent=2))
else:
    print(f"Error {resp.status_code}: {resp.text}")
    resp.raise_for_status()

print(f"\n🎉 Endpoint URL: {score_url}")